<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/09_transformer_timeseries.ipynb)


# colab — Transformer Time-Series Forecasting

## _One-Step-Ahead Prediction with a Small Transformer on a Free T4_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Forecast a synthetic time series one step ahead using a small
  Transformer encoder trained on a free Colab GPU.
- **Hardware**: Designed for Google Colab (T4 default). Auto-detected.
- **Data**: Fully synthetic — trend, seasonality, noise, and optional
  volatility clustering generated in the notebook.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Install PyTorch and Matplotlib.
2. **Synthetic Series**: Generate 5 000 observations (known structure).
3. **Windows**: Create rolling input-output windows for supervised learning.
4. **Model**: Define a tiny Transformer encoder (or LSTM fallback).
5. **Training**: Fit on GPU for a few epochs.
6. **Evaluation**: Plot predictions vs actual values on a held-out test set.
7. **Optional**: Repeat the pipeline on real SPY data via yfinance.

### Environment Setup

We verify the GPU and import the libraries needed for time-series
modelling and plotting.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install matplotlib numpy torch

In [ ]:
#@title Imports
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")
print(f"PyTorch version: {torch.__version__}")

### Synthetic Time Series

We generate a 5 000-point series with:

- a slow linear trend,
- a 50-period sine wave (seasonality),
- Gaussian noise,
- and an optional GARCH-like volatility cluster.

Because the ground truth is known, we can evaluate whether the model
learns the signal rather than over-fitting randomness.

In [ ]:
#@title Generate series
np.random.seed(42)
N = 50_000
t = np.arange(N, dtype=np.float32)

trend = 0.0002 * t
season = 0.02 * np.sin(2 * np.pi * t / 50)
noise = 0.01 * np.random.standard_normal(N)

# Optional GARCH-like volatility clustering
vol = np.zeros(N, dtype=np.float32)
vol[0] = 0.01
for i in range(1, N):
    vol[i] = np.sqrt(
        0.00001 + 0.85 * vol[i - 1] ** 2
        + 0.1 * noise[i - 1] ** 2
    )
returns = vol * np.random.standard_normal(N)

# Combine deterministic and stochastic components
signal = trend + season + returns
price = 100 * np.exp(np.cumsum(signal))

# We forecast the return series (easier to learn than raw price)
series = signal

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(series[:1000])
ax.set_title("First 1 000 observations of synthetic series")
ax.set_xlabel("Time step")
ax.set_ylabel("Return")
plt.show()

### Rolling Window Dataset

We create overlapping windows of length `lookback` and try to predict
the very next value. This turns the problem into a supervised regression
task.

In [ ]:
#@title Build windowed dataset
LOOKBACK = 64


class TimeSeriesDataset(Dataset):
    def __init__(self, data, lookback):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.lookback = lookback

    def __len__(self):
        return len(self.data) - self.lookback

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.lookback]
        y = self.data[idx + self.lookback]
        return x.unsqueeze(-1), y


# Train / val / test split
n = len(series)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_ds = TimeSeriesDataset(series[:train_end], LOOKBACK)
val_ds = TimeSeriesDataset(series[train_end:val_end], LOOKBACK)
test_ds = TimeSeriesDataset(series[val_end:], LOOKBACK)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print(f"Train: {len(train_ds):,}, Val: {len(val_ds):,}, "
      f"Test: {len(test_ds):,}")

### Small Transformer Encoder

A minimal Transformer: linear projection, a few encoder layers with
multi-head self-attention, and a small regression head that predicts
the next scalar value from the final hidden state.

In [ ]:
#@title Transformer model
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(
            0, max_len, dtype=torch.float32
        ).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TinyTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4,
                 num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            enc_layer, num_layers=num_layers
        )
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (batch, lookback, 1)
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        return self.head(x[:, -1, :]).squeeze(-1)


model = TinyTransformer().to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")

### Training Loop

We use MSE loss and Adam. A small learning rate keeps the Transformer
stable on this noisy signal.

In [ ]:
#@title Train on GPU
criterion = nn.MSELoss()
optim = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10
history = []

for ep in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    tr_n = 0
    t0 = time.perf_counter()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optim.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optim.step()
        tr_loss += loss.item() * xb.size(0)
        tr_n += xb.size(0)
    tr_loss /= tr_n

    model.eval()
    val_loss = 0.0
    val_n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * xb.size(0)
            val_n += xb.size(0)
    val_loss /= val_n

    dt = time.perf_counter() - t0
    history.append(
        {"epoch": ep, "tr": tr_loss, "val": val_loss}
    )
    print(f"Ep {ep:2d}/{EPOCHS}  train={tr_loss:.6f}  "
          f"val={val_loss:.6f}  ({dt:.1f} s)")

torch.cuda.synchronize() if device == "cuda" else None

### Evaluation: Predictions vs Actual

We run the trained model on the held-out test set and plot the
one-step-ahead forecasts against ground truth.

In [ ]:
#@title Plot test predictions
model.eval()
preds, actuals = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        preds.extend(pred)
        actuals.extend(yb.numpy())

preds = np.array(preds)
actuals = np.array(actuals)

mse = np.mean((preds - actuals) ** 2)
mae = np.mean(np.abs(preds - actuals))
print(f"Test MSE: {mse:.6f}, MAE: {mae:.6f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(actuals[:500], label="Actual", alpha=0.8)
ax.plot(preds[:500], label="Predicted", alpha=0.8)
ax.set_title("Test set: first 500 one-step-ahead forecasts")
ax.set_xlabel("Test time step")
ax.set_ylabel("Return")
ax.legend()
ax.grid(True)
plt.show()

### Optional: Repeat on Real Market Data

Uncomment the block below to download SPY closing prices via
yfinance and run the same forecasting pipeline. The results will be
noisier because real markets are far less predictable than our
synthetic toy.

In [ ]:
#@title Optional SPY pipeline (uncomment to run)
# !pip -q install yfinance
# import yfinance as yf
# spy = yf.download("SPY", start="2020-01-01",
#                   end="2025-01-01")
# spy_close = spy["Close"].dropna().values
# spy_returns = np.log(spy_close[1:] / spy_close[:-1])
# print(f"SPY observations: {len(spy_returns)}")
# # Re-run the windowed dataset and training cells above
# # with `series = spy_returns` to compare.

### Exercises

1. **Longer lookback**: Increase `LOOKBACK` to 128. Does the model
   capture the 50-period seasonality more accurately?
2. **More layers**: Increase `num_layers` to 4 or `d_model` to 128.
   Does validation loss improve or does the model overfit?
3. **LSTM comparison**: Swap the Transformer for an LSTM with the
   same parameter budget. Which converges faster?
4. **Real data**: Uncomment the yfinance block and train on SPY or
   BTC-USD returns. How much higher is the test MAE?
5. **Multi-step**: Modify the head to predict 5 steps ahead instead
   of 1. Does accuracy degrade gracefully?

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
